# C9-dimensionality-reduction — Practice p07 — Solution

In [1]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["glacier", "avalanche", "summit", "ridge", "valley",
         "melody", "rhythm", "chorus", "drummer", "saxophone"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
fro2 = float((W * W).sum())
assert np.isclose(fro2, 10.0, atol=1e-9, rtol=0)
U, s, Vt = np.linalg.svd(W, full_matrices=False)
W4 = U[:, :4] @ np.diag(s[:4]) @ Vt[:4]
err_direct = float(np.sqrt(((W - W4) ** 2).sum()))
err_tail = float(np.sqrt((s[4:] ** 2).sum()))
id_gap = float(abs(err_direct - err_tail))
assert id_gap < 1e-9
rel_err2 = float(err_direct**2 / fro2)

print("rank-4 error, direct / tail:", err_direct, err_tail)
print("relative squared error:", rel_err2)

rank-4 error, direct / tail: 1.604716413506125 1.6047164135061245
relative squared error: 0.25751147677759606


The unit-row stack has squared Frobenius norm $10$.  Its rank-$4$ residual norm is $1.6047164135$ by both direct measurement and the spectral tail, giving relative squared error $0.2575114768$.

### Answer check

In [2]:
assert W.shape == (10, 100)
assert W.dtype == np.float64
assert np.isclose(fro2, 10.0, atol=1e-9, rtol=0)
assert np.isclose(err_direct, 1.604716413506125, atol=1e-12, rtol=0)
assert id_gap < 1e-9
assert np.isclose(rel_err2, 0.25751147677759606, atol=1e-12, rtol=0)